In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("..")
from modeling import COVARIATES, oof_predict, prediction_metrics

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
TRAIN_DIR = DATA_DIR / "train"
PREVALENCE_MIN_COUNT = 200
META = ["participant_id", "cohort", "research_stage", "array_index", "collection_date"]


def read(path):
    df = pd.read_csv(path, low_memory=False)
    df["participant_id"] = df["participant_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    df.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", c) for c in df.columns]
    return df


diet = read(DATA_DIR / "diet_experiment" / "diet_feature_matrix_baseline_full.csv")
covariates = read(DATA_DIR / "covariates" / "covariates_age_sex_bmi_smoking_oralhealth_baseline.csv")
species = read(DATA_DIR / "oral_experiment" / "species_baseline_filtered.csv")
species_cols = [c for c in species.columns if c not in META]
abundance = species[species_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)

In [ ]:
prevalence_n = (abundance > 0).sum()
retained = prevalence_n[prevalence_n >= PREVALENCE_MIN_COUNT].index.tolist()
values = abundance[retained].astype(float)
for col in retained:
    values.loc[values[col] <= 0, col] = 0.5 * np.percentile(values.loc[values[col] > 0, col], 1)
species_log10 = pd.concat([species[["participant_id"]], np.log10(values.div(values.sum(axis=1), axis=0) * 1e6)], axis=1)
species_log10.to_csv(DATA_DIR / "oral_experiment" / "oral_species_relative_log10_baseline.csv", index=False)

summary = pd.DataFrame({"species": species_cols, "prevalence_n": prevalence_n.to_numpy(),
                        "prevalence": (abundance > 0).mean().to_numpy(), "kept_for_modeling": prevalence_n.ge(PREVALENCE_MIN_COUNT).to_numpy()})
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(TRAIN_DIR / "model_species_abundance_summary_baseline.csv", index=False)
print("retained species:", len(retained), "of", len(species_cols))

In [ ]:
diet_features = [c for c in diet.columns if c != "participant_id"]
model_input = (covariates[["participant_id"] + COVARIATES].merge(diet, on="participant_id")
               .merge(species_log10, on="participant_id"))
model_input.to_csv(TRAIN_DIR / "model_input_diet_cov_oral_species_abundance_baseline.csv", index=False)

models = {"covariates_only": COVARIATES, "covariates_plus_diet": COVARIATES + diet_features}
rows = []
for target in retained:
    y = model_input[target].to_numpy(dtype=float)
    for model, features in models.items():
        rows.append({"species": target, "model": model, "n": len(y),
                     **prediction_metrics(y, oof_predict(model_input[features], y))})
results = pd.DataFrame(rows)
results["delta_r"] = (results["pearson_r"] - results.groupby("species")["pearson_r"].transform("first")).where(results["model"] == "covariates_plus_diet")
results.to_csv(TRAIN_DIR / "model_results_diet_predict_oral_species_abundance_baseline.csv", index=False)
print(results[results["model"] == "covariates_plus_diet"].describe())